In [1]:
import tensorflow as tf
to_categorical = tf.keras.utils.to_categorical
array_to_img = tf.keras.preprocessing.image.array_to_img
ReduceLROnPlateau = tf.keras.callbacks.ReduceLROnPlateau
ImageDataGenerator = tf.keras.preprocessing.image.ImageDataGenerator
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objs as go
import plotly.figure_factory as ff
import plotly.express as px
import numpy as np
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
import pandas as pd

In [2]:
img_rows, img_cols = 128 , 128
batch_size = 16
epochs = 60
num_classes = 2
seed = 1

input_shape = (img_rows, img_cols, 3)
devide_list = tf.test.gpu_device_name()

In [3]:
datagen = ImageDataGenerator(rescale = 1./255,
                             horizontal_flip = True,
                             rotation_range=0.05)

val_test_datagen = ImageDataGenerator(rescale = 1./255)

In [4]:
train_it = datagen.flow_from_directory("D:/Portfolio/Academic Projects (June 2020 - June 2024)/05_Year-3_Perfume_Authenticity_CNN/dataset/Jomalone/train",
                                       target_size=(img_rows, img_cols),
                                       batch_size=batch_size,
                                       class_mode='binary',
                                       shuffle=True,
                                       color_mode='rgb',
                                       seed=seed)
test_it = datagen.flow_from_directory("D:/Portfolio/Academic Projects (June 2020 - June 2024)/05_Year-3_Perfume_Authenticity_CNN/dataset/Jomalone/test",
                                               target_size=(img_rows, img_cols),
                                               batch_size=batch_size,
                                               class_mode='binary',
                                               shuffle=False,
                                               color_mode='rgb',
                                               seed=seed)
val_it = val_test_datagen.flow_from_directory("D:/Portfolio/Academic Projects (June 2020 - June 2024)/05_Year-3_Perfume_Authenticity_CNN/dataset/Jomalone/val",
                                              target_size=(img_rows, img_cols),
                                              batch_size=batch_size,
                                              class_mode='binary',
                                              shuffle=False,
                                              color_mode='rgb',
                                              seed=seed)

Found 280 images belonging to 2 classes.
Found 40 images belonging to 2 classes.
Found 80 images belonging to 2 classes.


In [5]:
batchX, batchY = train_it.next()
print('Batch shape=%s, min=%.3f, max=%.3f' % (batchX.shape, batchX.min(), batchX.max()))

Batch shape=(16, 128, 128, 3), min=0.000, max=1.000


In [6]:
labels = list(train_it.class_indices.keys())
print(labels)

['Fake', 'Real']


In [7]:
print(Counter(train_it.classes))

Counter({0: 140, 1: 140})


In [8]:
weights = compute_class_weight('balanced', classes =  np.unique(train_it.classes), y = train_it.classes)
print(weights)

[1. 1.]


In [9]:
cw = dict(zip(np.unique(train_it.classes), weights))
print(cw)

{0: 1.0, 1: 1.0}


In [10]:
#Feature Extraction
model = tf.keras.Sequential()
model.add(tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

model.add(tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

model.add(tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

model.add(tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

#Image Classification
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

In [11]:
STEP_SIZE_TRAIN=train_it.n//train_it.batch_size
STEP_SIZE_VALID=val_it.n//val_it.batch_size
STEP_SIZE_TEST=test_it.n//test_it.batch_size

Compile Model

In [12]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 126, 126, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 63, 63, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 61, 61, 32)        9248      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 30, 30, 32)       0         
 2D)                                                             
                                                                 
 conv2d_2 (Conv2D)           (None, 28, 28, 64)        18496     
                                                                 
 max_pooling2d_2 (MaxPooling  (None, 14, 14, 64)       0

In [13]:
learning_rate_reduction = ReduceLROnPlateau(monitor='val_loss', patience = 2, verbose=1,factor=0.1, min_lr=0.000001)
callbacks_list = [learning_rate_reduction]

In [14]:
his = model.fit(
        train_it,
        validation_data=val_it,
        epochs=epochs,
        class_weight=cw,
        callbacks=callbacks_list,
        shuffle=True,
        steps_per_epoch=STEP_SIZE_TRAIN,
        validation_steps=STEP_SIZE_VALID)

Epoch 1/60
17/17 [==============================] - 5s 268ms/step - loss: 0.6951 - accuracy: 0.4735 - val_loss: 0.6918 - val_accuracy: 0.5000 - lr: 0.0010
Epoch 2/60
17/17 [==============================] - 4s 249ms/step - loss: 0.6914 - accuracy: 0.5152 - val_loss: 0.6872 - val_accuracy: 0.5000 - lr: 0.0010
Epoch 3/60
17/17 [==============================] - 4s 240ms/step - loss: 0.6962 - accuracy: 0.5265 - val_loss: 0.6882 - val_accuracy: 0.5000 - lr: 0.0010
Epoch 4/60
17/17 [==============================] - 4s 242ms/step - loss: 0.6527 - accuracy: 0.6023 - val_loss: 0.5728 - val_accuracy: 0.7250 - lr: 0.0010
Epoch 5/60
17/17 [==============================] - 4s 247ms/step - loss: 0.6279 - accuracy: 0.6364 - val_loss: 0.6543 - val_accuracy: 0.6875 - lr: 0.0010
Epoch 6/60
17/17 [==============================] - 4s 242ms/step - loss: 0.6535 - accuracy: 0.6780 - val_loss: 0.5636 - val_accuracy: 0.7500 - lr: 0.0010
Epoch 7/60
17/17 [==============================] - 4s 237ms/step - lo

plot loss

In [15]:
h1 = go.Scatter(y=his.history['loss'], 
                    mode="lines",
                    line=dict(
                        width=2,
                        color='blue'),
                        name="loss"
                   )
h2 = go.Scatter(y=his.history['val_loss'], 
                    mode="lines",
                    line=dict(
                        width=2,
                        color='red'),
                        name="val_loss"
                   )
                   
data = [h1,h2]
layout1 = go.Layout(title='Loss',
                   xaxis=dict(title='epochs'),
                   yaxis=dict(title=''))
fig1 = go.Figure(data, layout=layout1)
plotly.offline.iplot(fig1)

plot accuracy

In [16]:
h1 = go.Scatter(y=his.history['accuracy'], 
                    mode="lines", line=dict(
                    width=2,
                    color='blue'),
                    name="acc"
                   )
h2 = go.Scatter(y=his.history['val_accuracy'], 
                    mode="lines", line=dict(
                    width=2,
                    color='red'),
                    name="val_acc"
                   )

data = [h1,h2]
layout1 = go.Layout(title='Accuracy',
                   xaxis=dict(title='epochs'),
                   yaxis=dict(title=''))
fig1 = go.Figure(data = data, layout=layout1)
plotly.offline.iplot(fig1)

In [17]:
score = model.evaluate(test_it,verbose=0)
print("Test Loss:",score[0])
print("Test Accuracy:",score[1])

Test Loss: 0.5287798643112183
Test Accuracy: 0.824999988079071


In [18]:
predicted_classes = (model.predict(test_it, verbose=1) > 0.5).astype("int32")[:,0]

3/3 [==============================] - 1s 263ms/step


In [19]:
predicted_classes[:10]

array([1, 0, 0, 0, 0, 0, 0, 1, 0, 0])

In [20]:
test_it.classes[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [21]:
def cm_plot(cm, labels):
    x = labels
    y = labels

    z_text = [[str(y) for y in x] for x in cm]
    fig = ff.create_annotated_heatmap(cm, x=x, y=y, annotation_text=z_text, colorscale='blues')

    fig.update_layout(title_text='Confusion Matrix')

    fig.add_annotation(dict(font=dict(color="black",size=13),
                            x=0.5,
                            y=-0.15,
                            showarrow=False,
                            text="Predicted Value",
                            xref="paper",
                            yref="paper"
                           ))

    fig.add_annotation(dict(font=dict(color="black",size=13),
                            x=-0.20,
                            y=0.5,
                            showarrow=False,
                            text="Real Value",
                            textangle=-90,
                            xref="paper",
                            yref="paper"
                           ))

    fig.update_layout(margin=dict(t=50, l=200))
    fig['layout']['yaxis']['autorange'] = "reversed"

    fig['data'][0]['showscale'] = True
    fig.show()

In [22]:
Counter(test_it.classes)

Counter({0: 20, 1: 20})

In [23]:
cm = confusion_matrix(test_it.classes, predicted_classes)
cm

array([[17,  3],
       [ 5, 15]], dtype=int64)

In [24]:
cm_plot(cm, labels)